# Benford's Law for Audit Analytics

### Fraud Detection Aml Analytics — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), introductory optimisation concepts, and basic understanding of fraud detection and anti-money laundering terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe the information-extraction task and its relevance to fraud detection and anti-money laundering.
- Implement a rule-based extraction pipeline and evaluate accuracy on synthetic examples.
- Assess limitations of rule-based extraction and when ML-based approaches would be more appropriate.

**Estimated time:** 35–45 minutes

**Collection context:** Fraud detection, AML compliance, anomaly detection, and financial crime analytics in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** In naturally occurring datasets (like stock prices or account balances), the first digit is much more likely to be 1 than 9. Humans, when faking numbers, tend to use digits uniformly. Benford's Law is a standard tool used by IRS and bank auditors to catch fraud.

**Input data used:** A list of transaction amounts or balances.

**What we detect:** Inconsistencies between the actual first-digit distribution and Benford's expected distribution.

**Method used:** First-digit analysis and Chi-Square goodness-of-fit test.

**Learning goal:** Understand how simple mathematical laws can be used as powerful forensic tools.

**Primary stakeholders:** fraud analysts, compliance officers, risk managers, and financial crime investigators.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chisquare

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Benford's Distribution

The expected probability for the first digit $d \in \{1, \dots, 9\}$ is: 
$P(d) = \log_{10}(1 + 1/d)$

In [ ]:
benford_probs = [np.log10(1 + 1/d) for d in range(1, 10)]
digits = range(1, 10)

plt.bar(digits, benford_probs, color='skyblue')
plt.title("Expected Benford's Law Distribution")
plt.xlabel("First Digit")
plt.ylabel("Probability")
plt.xticks(digits)
plt.tight_layout()
plt.show()
plt.close()

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Natural data (like account balances) often follows a log-normal distribution, which satisfies Benford's Law. Manipulated data often has "invented" numbers.

In [ ]:
n = 2000

# Natural Data: Log-normal balances
natural_balances = RNG.lognormal(mean=10, sigma=2, size=n)

# Manipulated Data: Uniformly distributed (faked)
manipulated_balances = RNG.uniform(10, 100000, size=n)

def get_first_digit_counts(data):
    first_digits = [int(str(abs(x)).replace('.', '').lstrip('0')[0]) for x in data]
    return pd.Series(first_digits).value_counts(normalize=True).sort_index()

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: manual extraction (no automation) ---
print("Baseline: without automation, each document must be read and keyed manually.")
print("Any automated approach that achieves >80% field accuracy saves significant time.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
natural_counts = get_first_digit_counts(natural_balances)
manipulated_counts = get_first_digit_counts(manipulated_balances)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.bar(digits, natural_counts, alpha=0.7, label='Natural')
plt.step(digits, benford_probs, where='mid', color='#E76F51', label='Benford')
plt.title('Natural Account Balances')
plt.legend()

plt.subplot(1, 2, 2)
plt.bar(digits, manipulated_counts, alpha=0.7, color='orange', label='Manipulated')
plt.step(digits, benford_probs, where='mid', color='#E76F51', label='Benford')
plt.title('Manipulated (Faked) Data')
plt.legend()

plt.tight_layout()
plt.show()
plt.close()

# Statistical Test (Chi-Square)
chi_stat, p_val = chisquare(natural_counts, f_exp=benford_probs)
print(f"Natural Data: p-value = {p_val:.4f} (High p-value means it matches Benford)")

chi_stat_m, p_val_m = chisquare(manipulated_counts, f_exp=benford_probs)
print(f"Manipulated Data: p-value = {p_val_m:.4f} (Low p-value means it is likely fake)")

**Alt-text:** Bar chart titled "Natural Account Balances" and "Manipulated (Faked) Data". The chart ranks features or categories by magnitude to highlight dominant factors.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary of Results:**
- The **Natural Data** has a p-value > 0.05, meaning we cannot reject the idea that it follows Benford's Law. This is expected for authentic financial data.
- The **Manipulated Data** has a very small p-value, clearly showing it does not follow the expected mathematical pattern. This is a "red flag" that would lead an auditor to investigate these accounts further.

**Why this is used in Banking:** Auditors apply Benford's Law to transaction logs, expense reports, and tax filings to detect systemic fraud or manual manipulation of records.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: process a new document ---
print("Operational demonstration — field extraction:\n")
new_doc = "Invoice #2055. Vendor: Wayne Enterprises. Date: 2024-06-15. Total: $8750.00"
result = extract_info(new_doc)
print(f"  Raw: {new_doc}")
print(f"  Vendor: {result[0]}  Date: {result[1]}  Amount: ${result[2]:,.2f}")
print("\nExtracted fields would auto-populate the accounts-payable system.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end fraud detection and anti-money laundering workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- False positives can freeze legitimate accounts, causing financial hardship and customer distrust.
- Fraud models risk encoding biases against specific demographic groups or geographic regions.
- Transaction monitoring must balance fraud prevention with customer privacy rights.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in fraud detection and anti-money laundering?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.